# 🍌🍳🥑 Compare posteriors.

This notebook loads up posterior distributions for different shoreline options and visualizes them with corner plots. First, we'll copy what we want to be the default posterior to its own special file that will be easy to share.

In [ ]:
!cp posteriors/all+no-magma+uncertainties=True+numpyro.nc upload-to-zenodo/cosmic-shoreline-btwm2026-posterior.nc
!cp upload-to-zenodo/cosmic-shoreline-btwm2026-posterior.nc .

Next, we'll load the posteriors and write them out into a latex-friendly form that we can paste into the manuscript.

In [ ]:
from shoreline import * 

In [ ]:
import arviz as az 
import corner 

print('Here are the available posteriors:\n')
posteriors = {}
for f in glob.glob('posteriors/*uncertainties=*+numpyro.nc'):
    k = f.split('/')[1].split('+numpyro.nc')[0]
    print(f' {k}')
    posteriors[k] = az.from_netcdf(f)
    

Print out parameter values + confidence intervals.

In [ ]:
!mkdir latex-posteriors/

In [ ]:
warnings.simplefilter('ignore')

with open('latex-posteriors/posteriors-shorelines.tex', 'w') as f:

    for fluxlimit, inference in posteriors.items():
        f.write(f'%{fluxlimit}\n')
        po = inference.posterior
        po['fo'] = 10**po['log_f_0']
        po['logL_nohz'] = -po['log_f_0']/po['q']
        po['L_nohz'] = 10**po['logL_nohz']
        po['L_nohz'] = 10**po['logL_nohz']
        po['wninetyfive'] = 5.89*po['w']

        po['wninetyfive-as-flux-factor'] = 10**po['wninetyfive']
        po['q-from-ard'] = - po['log_f_0']/(-1.7)

        lines = latexify_arviz_posterior(po, label=fluxlimit.replace('uncertainties=True', ''))
        #for l in lines:
        #    print(l)
        f.writelines(lines)

In [ ]:
from scipy.special import erf
levels = erf(np.arange(1,3)/np.sqrt(2))
levels

Finally, we'll make some corner plots visualizing all the posteriors and comparing them to each other.

In [ ]:
plt.close("all")
figsize=(6,6)
question = {"any": "The cosmic shoreline\n(any temperature)", 
            "no-magma": "The cosmic shoreline\n(excluding magma oceans)",
            "no-magma-no-freeze": "The cosmic shoreline\n(exluding magma oceans\nand $\sf CO_2$ freezeout)"}
better_titles = ["$\sf \log_{10}f_0/f_\oplus$", "$\sf p$", "$\sf q$", "$\sf \ln w$"]
var_names = ["log_f_0", "p", "q", "ln_w"] 

kw = dict(
    var_names=var_names,
    labels=better_titles,
    bins=40,
    levels=levels,
    plot_density=False,
    plot_datapoints=False,
    show_titles=True,
    title_kwargs=dict(fontsize=9),
    range=[[-1, 6], [0, 10], [0, 3], [-6, 2]],
)
for fluxlimit in ["any", "no-magma", "no-magma-no-freeze"]:
    labels = {
        "all": f"Exoplanets + Solar System",
        "exo": "Exoplanets Only",
        "solar": "Solar System Only",
    }
    fig = plt.figure(figsize=figsize)
    colors = {
        f"solar+{fluxlimit}": "mediumslateblue",
        f"exo+{fluxlimit}": "tomato",
        f"all+{fluxlimit}": "black",
    }
    for k in colors:
        if "all" in k:
            alpha = 1
        else:
            alpha = 0.5
        corner.corner(
            posteriors[f'{k}+uncertainties=True'],
            color=colors[k],
            hist_kwargs=dict(density=True, label=labels[k.split("+")[0]]),
            fig=fig,
            contour_kwargs=dict(alpha=alpha, color=colors[k]),
            **kw,
        )
    plt.sca(fig.get_axes()[1])
    plt.title(question[fluxlimit], ha='left', va='center', y=0.5)
    plt.sca(fig.get_axes()[5])
    plt.legend(bbox_to_anchor=(1, 1), frameon=False)
    plt.savefig(f"figures/posteriors-{fluxlimit}.pdf")

In [ ]:
!cp figures/posteriors-no-magma.pdf paper-figures/.

Let's make a plot comparing the posteriors from three different flux limits.

In [ ]:
fig = plt.figure(figsize=figsize)
subset = 'all'
for fluxlimit in [ "any", "no-magma-no-freeze", "no-magma"]:

    fluxlimit_label = {"any": 'any temperature', 
                  "no-magma":'no magma ocean',
                  "no-magma-no-freeze":'no magma ocean,\nno $\sf CO_2$ freezeout'}
    uncertainty_labels = {True:f'with uncertainties', 
                          False:f'without uncertainties'}
    
    colors = {f"any":"skyblue",  "no-magma":"black", "no-magma-no-freeze": "orchid"}
    for u in [True]:
        alpha = {True: 1.0, False: 0.3}[u]
        color_kwargs = dict(alpha=alpha, color=colors[fluxlimit])
        corner.corner(
            posteriors[f"{subset}+{fluxlimit}+uncertainties={u}"],
            color=colors[fluxlimit],
            contour_kwargs=color_kwargs,
            hist_kwargs=color_kwargs | dict(density=True, label=f'{fluxlimit_label[fluxlimit]}'),
            fig=fig,
            **(kw | dict(show_titles = False)),
        )
    plt.sca(fig.get_axes()[0])
    plt.legend(bbox_to_anchor=(1.2, 1), loc='upper left', frameon=False)
    plt.savefig('figures/posteriors-compare-fluxlimits.pdf')

In [ ]:
!cp figures/posteriors-compare-fluxlimits.pdf paper-figures/.

Let's make a plot comparing the results if we do and don't include uncertainties.

In [ ]:
fig = plt.figure(figsize=figsize)
subset = 'all'
for fluxlimit in [ "no-magma-no-freeze", "no-magma"]:

    fluxlimit_label = {"any": 'any temperature', 
                  "no-magma":'no magma ocean',
                  "no-magma-no-freeze":'no magma ocean,\nno $\sf CO_2$ freezeout'}
    uncertainty_labels = {True:f'with planet uncertainties', 
                          False:f'without planet uncertainties'}
    
    colors = {f"any":"green",  "no-magma":"black", "no-magma-no-freeze": "orchid"}
    for u in [True, False]:
        alpha = {True: 1.0, False: 0.3}[u]
        color_kwargs = dict(alpha=alpha, color=colors[fluxlimit])
        corner.corner(
            posteriors[f"{subset}+{fluxlimit}+uncertainties={u}"],
            color=colors[fluxlimit],
            contour_kwargs=color_kwargs,
            hist_kwargs=color_kwargs | dict(density=True, label=f'{fluxlimit_label[fluxlimit]},\n{uncertainty_labels[u]}'),
            fig=fig,
            **(kw | dict(show_titles = False)),
        )
    plt.sca(fig.get_axes()[0])
    plt.legend(bbox_to_anchor=(2.2, 1), loc='upper left', frameon=False)
    plt.savefig('figures/posteriors-with-and-without+uncertainties.pdf')

In [ ]:
!cp figures/posteriors-with-and-without+uncertainties.pdf paper-figures/